In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
!find /usr -name "libcudart.so*" 2>/dev/null | head -20

In [ ]:
!find /usr -name "cuda_runtime.h" 2>/dev/null | head -20

In [ ]:
!gcc --version

In [ ]:
!ldconfig -p | grep -E 'libcuda|libnvrtc|libcudart'

In [ ]:
!which conda

In [ ]:
!conda list | grep -Ei 'cuda|cudatoolkit|nvrtc|cupy|numba'

In [ ]:
!python -c "import cupy; print('CuPy:', cupy.__version__); print('CUDA:', cupy.cuda.runtime.runtimeGetVersion())"

In [ ]:
!echo $CONDA_PREFIX

In [ ]:
!find $CONDA_PREFIX -name "libnvrtc.so*" 2>/dev/null

In [ ]:
!find $CONDA_PREFIX -name "libcudart.so*" 2>/dev/null

In [ ]:
!find $CONDA_PREFIX -name "cuda_runtime.h" 2>/dev/null

In [ ]:
!which python

In [ ]:
!python --version

In [ ]:
!find $CONDA_PREFIX -name "nvrtc.h" -o -name "cuda.h" 2>/dev/null

In [ ]:
!find $CONDA_PREFIX -name "libcuda.so*" -o -name "libnvrtc.so*" -o -name "libcudart.so*" 2>/dev/null

In [ ]:
#include <stdio.h>
#include <dlfcn.h>

int main2() {
    void *h = dlopen("libnvrtc.so.12", RTLD_NOW);

    if (!h) {
        printf("FAILED: %s\n", dlerror());
        return 1;
    }

    printf("SUCCESS: NVRTC loaded!\n");

    dlclose(h);
    return 0;
}

In [ ]:
main2()

In [ ]:

int main3() {
    const char *path =
        "/srv/conda/envs/notebook/lib/python3.11/site-packages/"
        "nvidia/cuda_nvrtc/lib/libnvrtc.so.12";

    void *h = dlopen(path, RTLD_NOW);

    if (!h) {
        printf("FAILED: %s\n", dlerror());
        return 1;
    }

    printf("SUCCESS: NVRTC loaded!\n");

    dlclose(h);
    return 0;
}

In [ ]:
main3()

In [ ]:


int main4() {
    void *h = dlopen("libcuda.so.1", RTLD_NOW);

    if (!h) {
        printf("FAILED: %s\n", dlerror());
        return 1;
    }

    printf("SUCCESS: CUDA driver loaded!\n");

    dlclose(h);
    return 0;
}

In [ ]:
main4()

In [ ]:


typedef int CUresult;
typedef int CUdevice;

#define CUDA_SUCCESS 0

int gpu_test2()
{
    void *cuda = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda) {
        printf("FAILED to load CUDA driver: %s\n", dlerror());
        return 1;
    }

    typedef CUresult (*cuInit_t)(unsigned int);
    typedef CUresult (*cuDeviceGetCount_t)(int *);
    typedef CUresult (*cuDeviceGet_t)(CUdevice *, int);
    typedef CUresult (*cuDeviceGetName_t)(char *, int, CUdevice);

    cuInit_t cuInit =
        (cuInit_t)dlsym(cuda, "cuInit");

    cuDeviceGetCount_t cuDeviceGetCount =
        (cuDeviceGetCount_t)dlsym(cuda, "cuDeviceGetCount");

    cuDeviceGet_t cuDeviceGet =
        (cuDeviceGet_t)dlsym(cuda, "cuDeviceGet");

    cuDeviceGetName_t cuDeviceGetName =
        (cuDeviceGetName_t)dlsym(cuda, "cuDeviceGetName");

    if (!cuInit || !cuDeviceGetCount ||
        !cuDeviceGet || !cuDeviceGetName) {

        printf("FAILED to find CUDA driver functions\n");
        dlclose(cuda);
        return 1;
    }

    CUresult r = cuInit(0);

    if (r != CUDA_SUCCESS) {
        printf("cuInit FAILED: %d\n", r);
        dlclose(cuda);
        return 1;
    }

    int count = 0;

    r = cuDeviceGetCount(&count);

    if (r != CUDA_SUCCESS) {
        printf("cuDeviceGetCount FAILED: %d\n", r);
        dlclose(cuda);
        return 1;
    }

    printf("CUDA devices visible: %d\n", count);

    for (int i = 0; i < count; i++) {

        CUdevice dev;
        char name[256];

        cuDeviceGet(&dev, i);
        cuDeviceGetName(name, sizeof(name), dev);

        printf("GPU %d: %s\n", i, name);
    }

    dlclose(cuda);

    return 0;
}

In [ ]:
gpu_test2()

In [ ]:


// #include <string.h>


// typedef int CUresult;
// typedef int CUdevice;
// typedef void* CUcontext;
// typedef void* CUmodule;
// typedef void* CUfunction;
// typedef unsigned long long CUdeviceptr;

// #define CUDA_SUCCESS 0

// typedef CUresult (*PFN_cuInit)(unsigned int);
// typedef CUresult (*PFN_cuDeviceGetCount)(int *);
// typedef CUresult (*PFN_cuDeviceGet)(CUdevice *, int);
// typedef CUresult (*PFN_cuDeviceGetName)(char *, int, CUdevice);
// typedef CUresult (*PFN_cuCtxCreate)(CUcontext *, unsigned int, CUdevice);
// typedef CUresult (*PFN_cuMemAlloc)(CUdeviceptr *, size_t);
// typedef CUresult (*PFN_cuMemFree)(CUdeviceptr);
// typedef CUresult (*PFN_cuMemcpyHtoD)(CUdeviceptr, const void *, size_t);
// typedef CUresult (*PFN_cuMemcpyDtoH)(void *, CUdeviceptr, size_t);
// typedef CUresult (*PFN_cuModuleLoadData)(CUmodule *, const void *);
// typedef CUresult (*PFN_cuModuleGetFunction)(CUfunction *, CUmodule, const char *);
// typedef CUresult (*PFN_cuLaunchKernel)(
//     CUfunction,
//     unsigned int, unsigned int, unsigned int,
//     unsigned int, unsigned int, unsigned int,
//     unsigned int,
//     void *,
//     void **,
//     void **
// );
// typedef CUresult (*PFN_cuCtxSynchronize)(void);
// typedef CUresult (*PFN_cuModuleUnload)(CUmodule);
// typedef CUresult (*PFN_cuCtxDestroy)(CUcontext);

// #define LOAD(name) do { \
//     name = (PFN_##name)dlsym(cuda, #name "_v2"); \
//     if (!name) name = (PFN_##name)dlsym(cuda, #name); \
//     if (!name) { printf("Missing %s\n", #name); return 1; } \
// } while(0)

// int gpu_test()
// {
//     void *cuda = dlopen("libcuda.so.1", RTLD_NOW);

//     if (!cuda) {
//         printf("CUDA driver load failed: %s\n", dlerror());
//         return 1;
//     }

//     PFN_cuInit cuInit;
//     PFN_cuDeviceGetCount cuDeviceGetCount;
//     PFN_cuDeviceGet cuDeviceGet;
//     PFN_cuDeviceGetName cuDeviceGetName;
//     PFN_cuCtxCreate cuCtxCreate;
//     PFN_cuMemAlloc cuMemAlloc;
//     PFN_cuMemFree cuMemFree;
//     PFN_cuMemcpyHtoD cuMemcpyHtoD;
//     PFN_cuMemcpyDtoH cuMemcpyDtoH;
//     PFN_cuModuleLoadData cuModuleLoadData;
//     PFN_cuModuleGetFunction cuModuleGetFunction;
//     PFN_cuLaunchKernel cuLaunchKernel;
//     PFN_cuCtxSynchronize cuCtxSynchronize;
//     PFN_cuModuleUnload cuModuleUnload;
//     PFN_cuCtxDestroy cuCtxDestroy;

//     LOAD(cuInit);
//     LOAD(cuDeviceGetCount);
//     LOAD(cuDeviceGet);
//     LOAD(cuDeviceGetName);
//     LOAD(cuCtxCreate);
//     LOAD(cuMemAlloc);
//     LOAD(cuMemFree);
//     LOAD(cuMemcpyHtoD);
//     LOAD(cuMemcpyDtoH);
//     LOAD(cuModuleLoadData);
//     LOAD(cuModuleGetFunction);
//     LOAD(cuLaunchKernel);
//     LOAD(cuCtxSynchronize);
//     LOAD(cuModuleUnload);
//     LOAD(cuCtxDestroy);

//     if (cuInit(0) != CUDA_SUCCESS) {
//         printf("cuInit failed\n");
//         return 1;
//     }

//     int count = 0;
//     cuDeviceGetCount(&count);

//     printf("CUDA devices: %d\n", count);

//     CUdevice dev;
//     cuDeviceGet(&dev, 0);

//     char name[256];
//     cuDeviceGetName(name, sizeof(name), dev);

//     printf("Using: %s\n", name);

//     CUcontext ctx;
//     cuCtxCreate(&ctx, 0, dev);

//     /*
//        Tiny GPU kernel.
//        Each GPU thread performs one addition.
//     */
//     const char *src =
//         "extern \"C\" __global__ "
//         "void add(float *a, float *b, float *c) {"
//         "int i = blockIdx.x * blockDim.x + threadIdx.x;"
//         "c[i] = a[i] + b[i];"
//         "}";

//     /*
//        Load NVRTC WITHOUT cuda.h or nvrtc.h.
//     */
//     void *nvrtc = dlopen(
//         "/srv/conda/envs/notebook/lib/python3.11/site-packages/"
//         "nvidia/cuda_nvrtc/lib/libnvrtc.so.12",
//         RTLD_NOW
//     );

//     if (!nvrtc) {
//         printf("NVRTC load failed: %s\n", dlerror());
//         return 1;
//     }

//     typedef int (*CreateProgram)(
//         void **, const char *, const char *,
//         int, const char **, const char **
//     );

//     typedef int (*CompileProgram)(
//         void *, int, const char **
//     );

//     typedef int (*GetPTXSize)(
//         void *, size_t *
//     );

//     typedef int (*GetPTX)(
//         void *, char *
//     );

//     typedef int (*GetLogSize)(
//         void *, size_t *
//     );

//     typedef int (*GetLog)(
//         void *, char *
//     );

//     typedef int (*DestroyProgram)(
//         void **
//     );

//     CreateProgram create =
//         (CreateProgram)dlsym(nvrtc, "nvrtcCreateProgram");

//     CompileProgram compile =
//         (CompileProgram)dlsym(nvrtc, "nvrtcCompileProgram");

//     GetPTXSize get_ptx_size =
//         (GetPTXSize)dlsym(nvrtc, "nvrtcGetPTXSize");

//     GetPTX get_ptx =
//         (GetPTX)dlsym(nvrtc, "nvrtcGetPTX");

//     GetLogSize get_log_size =
//         (GetLogSize)dlsym(nvrtc, "nvrtcGetProgramLogSize");

//     GetLog get_log =
//         (GetLog)dlsym(nvrtc, "nvrtcGetProgramLog");

//     DestroyProgram destroy =
//         (DestroyProgram)dlsym(nvrtc, "nvrtcDestroyProgram");

//     void *program;

//     int r = create(
//         &program,
//         src,
//         "add.cu",
//         0,
//         NULL,
//         NULL
//     );

//     if (r != 0) {
//         printf("NVRTC create failed: %d\n", r);
//         return 1;
//     }

//     const char *opts[] = {
//         "--gpu-architecture=compute_80"
//     };

//     r = compile(program, 1, opts);

//     if (r != 0) {

//         size_t log_size = 0;
//         get_log_size(program, &log_size);

//         char *log = malloc(log_size + 1);
//         get_log(program, log);
//         log[log_size] = '\0';

//         printf("NVRTC compile failed:\n%s\n", log);

//         free(log);
//         return 1;
//     }

//     size_t ptx_size;
//     get_ptx_size(program, &ptx_size);

//     char *ptx = malloc(ptx_size);
//     get_ptx(program, ptx);

//     CUmodule module;
//     CUfunction kernel;

//     if (cuModuleLoadData(&module, ptx) != CUDA_SUCCESS) {
//         printf("PTX load failed\n");
//         return 1;
//     }

//     if (cuModuleGetFunction(
//             &kernel,
//             module,
//             "add") != CUDA_SUCCESS) {

//         printf("Kernel lookup failed\n");
//         return 1;
//     }

//     /*
//        Very small test first.
//     */
//     const int N = 1024;

//     float *a = malloc(N * sizeof(float));
//     float *b = malloc(N * sizeof(float));
//     float *c = malloc(N * sizeof(float));

//     for (int i = 0; i < N; i++) {
//         a[i] = i;
//         b[i] = 2.0f * i;
//     }

//     CUdeviceptr da, db, dc;

//     cuMemAlloc(&da, N * sizeof(float));
//     cuMemAlloc(&db, N * sizeof(float));
//     cuMemAlloc(&dc, N * sizeof(float));

//     cuMemcpyHtoD(da, a, N * sizeof(float));
//     cuMemcpyHtoD(db, b, N * sizeof(float));

//     void *args[] = {
//         &da,
//         &db,
//         &dc
//     };

//     r = cuLaunchKernel(
//         kernel,
//         4, 1, 1,
//         256, 1, 1,
//         0,
//         NULL,
//         args,
//         NULL
//     );

//     if (r != CUDA_SUCCESS) {
//         printf("Kernel launch failed: %d\n", r);
//         return 1;
//     }

//     cuCtxSynchronize();

//     cuMemcpyDtoH(
//         c,
//         dc,
//         N * sizeof(float)
//     );

//     int errors = 0;

//     for (int i = 0; i < N; i++) {
//         float expected = a[i] + b[i];

//         if (c[i] != expected) {
//             errors++;
//         }
//     }

//     printf("\n============================\n");
//     printf("GPU COMPUTATION TEST\n");
//     printf("============================\n");
//     printf("N          : %d\n", N);
//     printf("GPU        : %s\n", name);
//     printf("Verification: %s\n",
//            errors == 0 ? "PASS" : "FAIL");
//     printf("============================\n");

//     cuMemFree(da);
//     cuMemFree(db);
//     cuMemFree(dc);

//     cuModuleUnload(module);
//     cuCtxDestroy(ctx);

//     destroy(&program);

//     dlclose(nvrtc);
//     dlclose(cuda);

//     free(ptx);
//     free(a);
//     free(b);
//     free(c);

//     return 0;
// }

// gpu_test();

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <dlfcn.h>
#include <sys/time.h>

#define BENCH_N 10000000

double bench_time()
{
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return (double)tv.tv_sec + (double)tv.tv_usec * 1.0e-6;
}

int gpu_benchmark2()
{
    /* =========================================================
       CUDA DRIVER API TYPES
       ========================================================= */

    typedef int CUresult;
    typedef int CUdevice;
    typedef void *CUcontext;
    typedef void *CUmodule;
    typedef void *CUfunction;
    typedef unsigned long long CUdeviceptr;

    #define CUDA_SUCCESS 0

    /* =========================================================
       LOAD CUDA DRIVER
       ========================================================= */

    void *cuda_lib = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda_lib) {
        printf("CUDA driver failed: %s\n", dlerror());
        return 1;
    }

    CUresult (*p_cuInit)(unsigned int);
    CUresult (*p_cuDeviceGetCount)(int *);
    CUresult (*p_cuDeviceGet)(CUdevice *, int);
    CUresult (*p_cuDeviceGetName)(char *, int, CUdevice);
    CUresult (*p_cuCtxCreate)(CUcontext *, unsigned int, CUdevice);
    CUresult (*p_cuMemAlloc)(CUdeviceptr *, size_t);
    CUresult (*p_cuMemFree)(CUdeviceptr);
    CUresult (*p_cuMemcpyHtoD)(CUdeviceptr, const void *, size_t);
    CUresult (*p_cuMemcpyDtoH)(void *, CUdeviceptr, size_t);
    CUresult (*p_cuModuleLoadData)(CUmodule *, const void *);
    CUresult (*p_cuModuleGetFunction)(CUfunction *, CUmodule, const char *);
    CUresult (*p_cuLaunchKernel)(
        CUfunction,
        unsigned int, unsigned int, unsigned int,
        unsigned int, unsigned int, unsigned int,
        unsigned int,
        void *, void **, void *
    );
    CUresult (*p_cuCtxSynchronize)(void);
    CUresult (*p_cuModuleUnload)(CUmodule);
    CUresult (*p_cuCtxDestroy)(CUcontext);

    #define LOAD2(fn)                                      \
        p_##fn = dlsym(cuda_lib, #fn "_v2");               \
        if (!p_##fn) p_##fn = dlsym(cuda_lib, #fn);        \
        if (!p_##fn) {                                    \
            printf("Missing CUDA function: %s\n", #fn);   \
            return 1;                                     \
        }

    LOAD2(cuInit)
    LOAD2(cuDeviceGetCount)
    LOAD2(cuDeviceGet)
    LOAD2(cuDeviceGetName)
    LOAD2(cuCtxCreate)
    LOAD2(cuMemAlloc)
    LOAD2(cuMemFree)
    LOAD2(cuMemcpyHtoD)
    LOAD2(cuMemcpyDtoH)
    LOAD2(cuModuleLoadData)
    LOAD2(cuModuleGetFunction)
    LOAD2(cuLaunchKernel)
    LOAD2(cuCtxSynchronize)
    LOAD2(cuModuleUnload)
    LOAD2(cuCtxDestroy)

    p_cuInit(0);

    int device_count = 0;
    p_cuDeviceGetCount(&device_count);

    printf("CUDA devices visible: %d\n", device_count);

    CUdevice device;
    p_cuDeviceGet(&device, 0);

    char gpu_name[256];
    p_cuDeviceGetName(gpu_name, 256, device);

    printf("Using: %s\n", gpu_name);

    CUcontext context;

    if (p_cuCtxCreate(&context, 0, device) != CUDA_SUCCESS) {
        printf("Context creation failed\n");
        return 1;
    }

    /* =========================================================
       HOST DATA
       ========================================================= */

    int n = BENCH_N;

    float *A = malloc((size_t)n * sizeof(float));
    float *B = malloc((size_t)n * sizeof(float));
    float *C = malloc((size_t)n * sizeof(float));
    float *G = malloc((size_t)n * sizeof(float));

    if (!A || !B || !C || !G) {
        printf("Host allocation failed\n");
        return 1;
    }

    for (int i = 0; i < n; i++) {
        A[i] = (float)i;
        B[i] = 2.0f * (float)i;
    }

    /* =========================================================
       CPU TEST
       ========================================================= */

    double t0 = bench_time();

    for (int i = 0; i < n; i++) {
        C[i] = A[i] + B[i];
    }

    double cpu_time = bench_time() - t0;

    /* =========================================================
       LOAD NVRTC
       ========================================================= */

    void *nvrtc_lib = dlopen(
        "/srv/conda/envs/notebook/lib/python3.11/"
        "site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12",
        RTLD_NOW
    );

    if (!nvrtc_lib) {
        printf("NVRTC failed: %s\n", dlerror());
        return 1;
    }

    typedef int (*nvrtcCreateProgram_t)(
        void **,
        const char *,
        const char *,
        int,
        const char **,
        const char **
    );

    typedef int (*nvrtcCompileProgram_t)(
        void *,
        int,
        const char **
    );

    typedef int (*nvrtcGetPTXSize_t)(
        void *,
        size_t *
    );

    typedef int (*nvrtcGetPTX_t)(
        void *,
        char *
    );

    typedef int (*nvrtcGetProgramLogSize_t)(
        void *,
        size_t *
    );

    typedef int (*nvrtcGetProgramLog_t)(
        void *,
        char *
    );

    typedef int (*nvrtcDestroyProgram_t)(
        void **
    );

    nvrtcCreateProgram_t p_create =
        dlsym(nvrtc_lib, "nvrtcCreateProgram");

    nvrtcCompileProgram_t p_compile =
        dlsym(nvrtc_lib, "nvrtcCompileProgram");

    nvrtcGetPTXSize_t p_get_ptx_size =
        dlsym(nvrtc_lib, "nvrtcGetPTXSize");

    nvrtcGetPTX_t p_get_ptx =
        dlsym(nvrtc_lib, "nvrtcGetPTX");

    nvrtcGetProgramLogSize_t p_get_log_size =
        dlsym(nvrtc_lib, "nvrtcGetProgramLogSize");

    nvrtcGetProgramLog_t p_get_log =
        dlsym(nvrtc_lib, "nvrtcGetProgramLog");

    nvrtcDestroyProgram_t p_destroy =
        dlsym(nvrtc_lib, "nvrtcDestroyProgram");

    if (!p_create || !p_compile || !p_get_ptx_size ||
        !p_get_ptx || !p_destroy) {

        printf("NVRTC functions missing\n");
        return 1;
    }

    /* =========================================================
       GPU KERNEL
       ========================================================= */

    const char *source =
        "extern \"C\" __global__ "
        "void add_arrays(float *a, float *b, float *c, int n) {"
        "    int i = blockIdx.x * blockDim.x + threadIdx.x;"
        "    if (i < n) c[i] = a[i] + b[i];"
        "}";

    void *program = NULL;

    if (p_create(
            &program,
            source,
            "add_arrays.cu",
            0,
            NULL,
            NULL) != 0) {

        printf("NVRTC program creation failed\n");
        return 1;
    }

    const char *options[] = {
        "--gpu-architecture=compute_80"
    };

    int compile_result =
        p_compile(program, 1, options);

    if (compile_result != 0) {

        printf("NVRTC compilation failed\n");

        if (p_get_log_size && p_get_log) {

            size_t log_size = 0;
            p_get_log_size(program, &log_size);

            char *log = malloc(log_size + 1);

            p_get_log(program, log);

            log[log_size] = '\0';

            printf("%s\n", log);

            free(log);
        }

        return 1;
    }

    size_t ptx_size = 0;

    p_get_ptx_size(program, &ptx_size);

    char *ptx = malloc(ptx_size);

    p_get_ptx(program, ptx);

    /* =========================================================
       LOAD GPU MODULE
       ========================================================= */

    CUmodule module;

    if (p_cuModuleLoadData(&module, ptx) != CUDA_SUCCESS) {
        printf("PTX module loading failed\n");
        return 1;
    }

    CUfunction kernel;

    p_cuModuleGetFunction(
        &kernel,
        module,
        "add_arrays"
    );

    /* =========================================================
       GPU MEMORY
       ========================================================= */

    CUdeviceptr dA;
    CUdeviceptr dB;
    CUdeviceptr dC;

    p_cuMemAlloc(
        &dA,
        (size_t)n * sizeof(float)
    );

    p_cuMemAlloc(
        &dB,
        (size_t)n * sizeof(float)
    );

    p_cuMemAlloc(
        &dC,
        (size_t)n * sizeof(float)
    );

    /* =========================================================
       HOST -> GPU
       ========================================================= */

    t0 = bench_time();

    p_cuMemcpyHtoD(
        dA,
        A,
        (size_t)n * sizeof(float)
    );

    p_cuMemcpyHtoD(
        dB,
        B,
        (size_t)n * sizeof(float)
    );

    p_cuCtxSynchronize();

    double h2d_time =
        bench_time() - t0;

    /* =========================================================
       GPU KERNEL
       ========================================================= */

    int threads = 256;
    int blocks =
        (n + threads - 1) / threads;

    void *kernel_args[] = {
        &dA,
        &dB,
        &dC,
        &n
    };

    t0 = bench_time();

    p_cuLaunchKernel(
        kernel,
        blocks, 1, 1,
        threads, 1, 1,
        0,
        NULL,
        kernel_args,
        NULL
    );

    p_cuCtxSynchronize();

    double gpu_kernel_time =
        bench_time() - t0;

    /* =========================================================
       GPU -> HOST
       ========================================================= */

    t0 = bench_time();

    p_cuMemcpyDtoH(
        G,
        dC,
        (size_t)n * sizeof(float)
    );

    p_cuCtxSynchronize();

    double d2h_time =
        bench_time() - t0;

    /* =========================================================
       VERIFICATION
       ========================================================= */

    int errors = 0;

    for (int i = 0; i < n; i++) {

        if (G[i] != C[i]) {

            errors++;

            if (errors <= 5) {
                printf(
                    "Mismatch %d: CPU=%f GPU=%f\n",
                    i,
                    C[i],
                    G[i]
                );
            }
        }
    }

    /* =========================================================
       RESULTS
       ========================================================= */

    printf("\n");
    printf("========================================\n");
    printf("GPU BENCHMARK\n");
    printf("========================================\n");

    printf("N               : %d\n", n);
    printf("GPU             : %s\n", gpu_name);

    printf("\n");

    printf(
        "CPU computation : %.6f s\n",
        cpu_time
    );

    printf(
        "GPU kernel      : %.6f s\n",
        gpu_kernel_time
    );

    printf(
        "CPU -> GPU      : %.6f s\n",
        h2d_time
    );

    printf(
        "GPU -> CPU      : %.6f s\n",
        d2h_time
    );

    double gpu_total =
        h2d_time +
        gpu_kernel_time +
        d2h_time;

    printf(
        "GPU total       : %.6f s\n",
        gpu_total
    );

    printf(
        "Kernel speedup  : %.2fx\n",
        cpu_time / gpu_kernel_time
    );

    printf(
        "Total speedup   : %.2fx\n",
        cpu_time / gpu_total
    );

    printf(
        "Verification    : %s\n",
        errors == 0 ? "PASS" : "FAIL"
    );

    printf("========================================\n");

    /* =========================================================
       CLEANUP
       ========================================================= */

    p_cuMemFree(dA);
    p_cuMemFree(dB);
    p_cuMemFree(dC);

    p_cuModuleUnload(module);
    p_cuCtxDestroy(context);

    p_destroy(&program);

    dlclose(nvrtc_lib);
    dlclose(cuda_lib);

    free(ptx);

    free(A);
    free(B);
    free(C);
    free(G);

    return 0;
}

gpu_benchmark2();

In [ ]:
#pragma cling add_include_path("/usr/local/cuda/include")
#pragma cling add_library_path("/usr/local/cuda/lib64")

In [ ]:
!which nvcc



In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>
#include <dlfcn.h>

/* ============================================================
   CUDA Driver API types/constants
   We avoid #include <cuda.h> because the Jupyter C23 parser
   does not have the CUDA headers in its include path.
   ============================================================ */

typedef int CUresult;
typedef int CUdevice;
typedef void *CUcontext;
typedef void *CUmodule;
typedef void *CUfunction;
typedef void *CUstream;
typedef unsigned long long CUdeviceptr;

#define CUDA_SUCCESS 0

#define CU_DEVICE_ATTRIBUTE_COMPUTE_CAPABILITY_MAJOR 75
#define CU_DEVICE_ATTRIBUTE_COMPUTE_CAPABILITY_MINOR 76

/* ============================================================
   Function-pointer types for CUDA Driver API
   ============================================================ */

typedef CUresult (*PFN_cuInit)(unsigned int);
typedef CUresult (*PFN_cuDeviceGetCount)(int *);
typedef CUresult (*PFN_cuDeviceGet)(CUdevice *, int);
typedef CUresult (*PFN_cuDeviceGetName)(char *, int, CUdevice);
typedef CUresult (*PFN_cuDeviceGetAttribute)(int *, int, CUdevice);

typedef CUresult (*PFN_cuCtxCreate)(CUcontext *, unsigned int, CUdevice);
typedef CUresult (*PFN_cuCtxDestroy)(CUcontext);

typedef CUresult (*PFN_cuMemAlloc)(CUdeviceptr *, size_t);
typedef CUresult (*PFN_cuMemFree)(CUdeviceptr);
typedef CUresult (*PFN_cuMemcpyHtoD)(CUdeviceptr, const void *, size_t);
typedef CUresult (*PFN_cuMemcpyDtoH)(void *, CUdeviceptr, size_t);

typedef CUresult (*PFN_cuModuleLoadData)(
    CUmodule *, const void *
);

typedef CUresult (*PFN_cuModuleGetFunction)(
    CUfunction *, CUmodule, const char *
);

typedef CUresult (*PFN_cuLaunchKernel)(
    CUfunction,
    unsigned int,
    unsigned int,
    unsigned int,
    unsigned int,
    unsigned int,
    unsigned int,
    unsigned int,
    CUstream,
    void **,
    void **
);

typedef CUresult (*PFN_cuCtxSynchronize)(void);

/* ============================================================
   NVRTC function types
   ============================================================ */

typedef int nvrtcResult;
typedef void *nvrtcProgram;

#define NVRTC_SUCCESS 0

typedef nvrtcResult (*PFN_nvrtcCreateProgram)(
    nvrtcProgram *,
    const char *,
    const char *,
    int,
    const char *const *,
    const char *const *
);

typedef nvrtcResult (*PFN_nvrtcCompileProgram)(
    nvrtcProgram,
    int,
    const char *const *
);

typedef nvrtcResult (*PFN_nvrtcGetPTXSize)(
    nvrtcProgram,
    size_t *
);

typedef nvrtcResult (*PFN_nvrtcGetPTX)(
    nvrtcProgram,
    char *
);

typedef nvrtcResult (*PFN_nvrtcGetProgramLogSize)(
    nvrtcProgram,
    size_t *
);

typedef nvrtcResult (*PFN_nvrtcGetProgramLog)(
    nvrtcProgram,
    char *
);

typedef nvrtcResult (*PFN_nvrtcDestroyProgram)(
    nvrtcProgram *
);

/* ============================================================
   LOAD FUNCTION
   ============================================================ */

#define LOAD(handle, name)                                      \
    do {                                                        \
        name = (typeof(name))dlsym(handle, #name);              \
        if (!(name)) {                                          \
            printf("FAILED loading %s\n", #name);               \
            return 1;                                           \
        }                                                       \
    } while (0)

/* ============================================================
   GPU TEST
   ============================================================ */

int gpu_compute_test()
{
    const long N = 10000000;

    void *cuda_lib = NULL;
    void *nvrtc_lib = NULL;

    /* --------------------------------------------------------
       Load CUDA Driver library
       -------------------------------------------------------- */

    cuda_lib = dlopen(
        "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
        RTLD_NOW
    );

    if (!cuda_lib) {
        printf("Could not load CUDA driver:\n%s\n", dlerror());
        return 1;
    }

    /* --------------------------------------------------------
       Load NVRTC from your actual conda environment
       -------------------------------------------------------- */

    nvrtc_lib = dlopen(
        "/srv/conda/envs/notebook/lib/python3.11/"
        "site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12",
        RTLD_NOW
    );

    if (!nvrtc_lib) {
        printf("Could not load NVRTC:\n%s\n", dlerror());
        dlclose(cuda_lib);
        return 1;
    }

    /* --------------------------------------------------------
       CUDA function pointers
       -------------------------------------------------------- */

    PFN_cuInit cuInit;
    PFN_cuDeviceGetCount cuDeviceGetCount;
    PFN_cuDeviceGet cuDeviceGet;
    PFN_cuDeviceGetName cuDeviceGetName;
    PFN_cuDeviceGetAttribute cuDeviceGetAttribute;

    PFN_cuCtxCreate cuCtxCreate;
    PFN_cuCtxDestroy cuCtxDestroy;

    PFN_cuMemAlloc cuMemAlloc;
    PFN_cuMemFree cuMemFree;

    PFN_cuMemcpyHtoD cuMemcpyHtoD;
    PFN_cuMemcpyDtoH cuMemcpyDtoH;

    PFN_cuModuleLoadData cuModuleLoadData;
    PFN_cuModuleGetFunction cuModuleGetFunction;

    PFN_cuLaunchKernel cuLaunchKernel;
    PFN_cuCtxSynchronize cuCtxSynchronize;

    /* --------------------------------------------------------
       Load CUDA functions
       -------------------------------------------------------- */

    cuInit = (PFN_cuInit)dlsym(cuda_lib, "cuInit");
    cuDeviceGetCount =
        (PFN_cuDeviceGetCount)dlsym(cuda_lib, "cuDeviceGetCount");
    cuDeviceGet =
        (PFN_cuDeviceGet)dlsym(cuda_lib, "cuDeviceGet");
    cuDeviceGetName =
        (PFN_cuDeviceGetName)dlsym(cuda_lib, "cuDeviceGetName");
    cuDeviceGetAttribute =
        (PFN_cuDeviceGetAttribute)
        dlsym(cuda_lib, "cuDeviceGetAttribute");

    cuCtxCreate =
        (PFN_cuCtxCreate)dlsym(cuda_lib, "cuCtxCreate_v2");

    if (!cuCtxCreate)
        cuCtxCreate =
            (PFN_cuCtxCreate)dlsym(cuda_lib, "cuCtxCreate");

    cuCtxDestroy =
        (PFN_cuCtxDestroy)dlsym(cuda_lib, "cuCtxDestroy_v2");

    if (!cuCtxDestroy)
        cuCtxDestroy =
            (PFN_cuCtxDestroy)dlsym(cuda_lib, "cuCtxDestroy");

    cuMemAlloc =
        (PFN_cuMemAlloc)dlsym(cuda_lib, "cuMemAlloc_v2");

    cuMemFree =
        (PFN_cuMemFree)dlsym(cuda_lib, "cuMemFree_v2");

    cuMemcpyHtoD =
        (PFN_cuMemcpyHtoD)dlsym(cuda_lib, "cuMemcpyHtoD_v2");

    cuMemcpyDtoH =
        (PFN_cuMemcpyDtoH)dlsym(cuda_lib, "cuMemcpyDtoH_v2");

    cuModuleLoadData =
        (PFN_cuModuleLoadData)dlsym(cuda_lib, "cuModuleLoadData");

    cuModuleGetFunction =
        (PFN_cuModuleGetFunction)
        dlsym(cuda_lib, "cuModuleGetFunction");

    cuLaunchKernel =
        (PFN_cuLaunchKernel)
        dlsym(cuda_lib, "cuLaunchKernel");

    cuCtxSynchronize =
        (PFN_cuCtxSynchronize)
        dlsym(cuda_lib, "cuCtxSynchronize");

    if (!cuInit ||
        !cuDeviceGetCount ||
        !cuDeviceGet ||
        !cuDeviceGetName ||
        !cuCtxCreate ||
        !cuCtxDestroy ||
        !cuMemAlloc ||
        !cuMemFree ||
        !cuMemcpyHtoD ||
        !cuMemcpyDtoH ||
        !cuModuleLoadData ||
        !cuModuleGetFunction ||
        !cuLaunchKernel ||
        !cuCtxSynchronize) {

        printf("ERROR: Could not load required CUDA functions.\n");

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    /* --------------------------------------------------------
       Load NVRTC functions
       -------------------------------------------------------- */

    PFN_nvrtcCreateProgram nvrtcCreateProgram;
    PFN_nvrtcCompileProgram nvrtcCompileProgram;
    PFN_nvrtcGetPTXSize nvrtcGetPTXSize;
    PFN_nvrtcGetPTX nvrtcGetPTX;
    PFN_nvrtcGetProgramLogSize nvrtcGetProgramLogSize;
    PFN_nvrtcGetProgramLog nvrtcGetProgramLog;
    PFN_nvrtcDestroyProgram nvrtcDestroyProgram;

    nvrtcCreateProgram =
        (PFN_nvrtcCreateProgram)
        dlsym(nvrtc_lib, "nvrtcCreateProgram");

    nvrtcCompileProgram =
        (PFN_nvrtcCompileProgram)
        dlsym(nvrtc_lib, "nvrtcCompileProgram");

    nvrtcGetPTXSize =
        (PFN_nvrtcGetPTXSize)
        dlsym(nvrtc_lib, "nvrtcGetPTXSize");

    nvrtcGetPTX =
        (PFN_nvrtcGetPTX)
        dlsym(nvrtc_lib, "nvrtcGetPTX");

    nvrtcGetProgramLogSize =
        (PFN_nvrtcGetProgramLogSize)
        dlsym(nvrtc_lib, "nvrtcGetProgramLogSize");

    nvrtcGetProgramLog =
        (PFN_nvrtcGetProgramLog)
        dlsym(nvrtc_lib, "nvrtcGetProgramLog");

    nvrtcDestroyProgram =
        (PFN_nvrtcDestroyProgram)
        dlsym(nvrtc_lib, "nvrtcDestroyProgram");

    if (!nvrtcCreateProgram ||
        !nvrtcCompileProgram ||
        !nvrtcGetPTXSize ||
        !nvrtcGetPTX ||
        !nvrtcGetProgramLogSize ||
        !nvrtcGetProgramLog ||
        !nvrtcDestroyProgram) {

        printf("ERROR: Could not load required NVRTC functions.\n");

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    /* ========================================================
       INITIALIZE CUDA
       ======================================================== */

    CUresult r;

    r = cuInit(0);

    if (r != CUDA_SUCCESS) {
        printf("cuInit FAILED: %d\n", r);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    int device_count = 0;

    r = cuDeviceGetCount(&device_count);

    if (r != CUDA_SUCCESS) {
        printf("cuDeviceGetCount FAILED: %d\n", r);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    printf("CUDA devices visible: %d\n", device_count);

    if (device_count == 0) {
        printf("No CUDA device found.\n");

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    /* ========================================================
       SELECT GPU 0
       ======================================================== */

    CUdevice dev;

    r = cuDeviceGet(&dev, 0);

    if (r != CUDA_SUCCESS) {
        printf("cuDeviceGet FAILED: %d\n", r);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    char name[256];

    cuDeviceGetName(
        name,
        sizeof(name),
        dev
    );

    printf("Using: %s\n", name);

    /* ========================================================
       CREATE CUDA CONTEXT
       ======================================================== */

    CUcontext ctx;

    r = cuCtxCreate(
        &ctx,
        0,
        dev
    );

    if (r != CUDA_SUCCESS) {
        printf("cuCtxCreate FAILED: %d\n", r);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    /* ========================================================
       CUDA KERNEL
       
       Simple vector operation:
       
           C[i] = A[i] * B[i] + A[i]
       
       This is deliberately simple.
       We first verify that GPU execution is correct.
       ======================================================== */

    const char *kernel_source =
        "extern \"C\" __global__ "
        "void vector_kernel("
        "const double *A, "
        "const double *B, "
        "double *C, "
        "long N)"
        "{"
        "    long i = "
        "        (long)blockIdx.x * blockDim.x "
        "        + threadIdx.x;"

        "    if (i < N)"
        "        C[i] = A[i] * B[i] + A[i];"
        "}";

    /* ========================================================
       COMPILE KERNEL WITH NVRTC
       ======================================================== */

    nvrtcProgram program;

    nvrtcResult nr;

    nr = nvrtcCreateProgram(
        &program,
        kernel_source,
        "vector_kernel.cu",
        0,
        NULL,
        NULL
    );

    if (nr != NVRTC_SUCCESS) {
        printf(
            "nvrtcCreateProgram FAILED: %d\n",
            nr
        );

        cuCtxDestroy(ctx);
        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    const char *options[] = {
        "--gpu-architecture=compute_80"
    };

    nr = nvrtcCompileProgram(
        program,
        1,
        options
    );

    if (nr != NVRTC_SUCCESS) {

        size_t log_size = 0;

        nvrtcGetProgramLogSize(
            program,
            &log_size
        );

        char *log =
            (char *)malloc(log_size + 1);

        nvrtcGetProgramLog(
            program,
            log
        );

        log[log_size] = '\0';

        printf(
            "NVRTC COMPILATION FAILED:\n%s\n",
            log
        );

        free(log);

        nvrtcDestroyProgram(&program);

        cuCtxDestroy(ctx);
        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    printf("NVRTC compilation: PASS\n");

    /* ========================================================
       GET PTX
       ======================================================== */

    size_t ptx_size = 0;

    nr = nvrtcGetPTXSize(
        program,
        &ptx_size
    );

    if (nr != NVRTC_SUCCESS) {
        printf("nvrtcGetPTXSize FAILED\n");

        nvrtcDestroyProgram(&program);
        cuCtxDestroy(ctx);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    char *ptx =
        (char *)malloc(ptx_size);

    nr = nvrtcGetPTX(
        program,
        ptx
    );

    if (nr != NVRTC_SUCCESS) {
        printf("nvrtcGetPTX FAILED\n");

        free(ptx);

        nvrtcDestroyProgram(&program);
        cuCtxDestroy(ctx);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    nvrtcDestroyProgram(&program);

    /* ========================================================
       LOAD PTX INTO CUDA
       ======================================================== */

    CUmodule module;

    r = cuModuleLoadData(
        &module,
        ptx
    );

    if (r != CUDA_SUCCESS) {

        printf(
            "cuModuleLoadData FAILED: %d\n",
            r
        );

        free(ptx);

        cuCtxDestroy(ctx);
        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    free(ptx);

    /* ========================================================
       GET KERNEL FUNCTION
       ======================================================== */

    CUfunction kernel;

    r = cuModuleGetFunction(
        &kernel,
        module,
        "vector_kernel"
    );

    if (r != CUDA_SUCCESS) {

        printf(
            "cuModuleGetFunction FAILED: %d\n",
            r
        );

        cuCtxDestroy(ctx);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    /* ========================================================
       HOST MEMORY
       ======================================================== */

    double *A =
        (double *)malloc(N * sizeof(double));

    double *B =
        (double *)malloc(N * sizeof(double));

    double *C =
        (double *)malloc(N * sizeof(double));

    if (!A || !B || !C) {

        printf("Host allocation FAILED\n");

        free(A);
        free(B);
        free(C);

        cuCtxDestroy(ctx);

        dlclose(nvrtc_lib);
        dlclose(cuda_lib);

        return 1;
    }

    for (long i = 0; i < N; i++) {

        A[i] = 2.0;
        B[i] = 3.0;
        C[i] = 0.0;
    }

    /* ========================================================
       GPU MEMORY
       ======================================================== */

    CUdeviceptr dA;
    CUdeviceptr dB;
    CUdeviceptr dC;

    size_t bytes =
        N * sizeof(double);

    r = cuMemAlloc(
        &dA,
        bytes
    );

    if (r != CUDA_SUCCESS) {
        printf("cuMemAlloc A FAILED: %d\n", r);
        return 1;
    }

    r = cuMemAlloc(
        &dB,
        bytes
    );

    if (r != CUDA_SUCCESS) {
        printf("cuMemAlloc B FAILED: %d\n", r);
        return 1;
    }

    r = cuMemAlloc(
        &dC,
        bytes
    );

    if (r != CUDA_SUCCESS) {
        printf("cuMemAlloc C FAILED: %d\n", r);
        return 1;
    }

    /* ========================================================
       HOST -> GPU
       ======================================================== */

    clock_t t0 = clock();

    cuMemcpyHtoD(
        dA,
        A,
        bytes
    );

    cuMemcpyHtoD(
        dB,
        B,
        bytes
    );

    clock_t t1 = clock();

    double h2d_time =
        (double)(t1 - t0)
        / CLOCKS_PER_SEC;

    /* ========================================================
       LAUNCH CONFIGURATION
       ======================================================== */

    unsigned int block_size = 256;

    unsigned int grid_size =
        (unsigned int)
        ((N + block_size - 1)
        / block_size);

    void *args[] = {
        &dA,
        &dB,
        &dC,
        (void *)&N
    };

    /* ========================================================
       GPU KERNEL TIMING
       ======================================================== */

    t0 = clock();

    r = cuLaunchKernel(
        kernel,

        grid_size,
        1,
        1,

        block_size,
        1,
        1,

        0,
        NULL,

        args,
        NULL
    );

    if (r != CUDA_SUCCESS) {

        printf(
            "cuLaunchKernel FAILED: %d\n",
            r
        );

        return 1;
    }

    r = cuCtxSynchronize();

    if (r != CUDA_SUCCESS) {

        printf(
            "cuCtxSynchronize FAILED: %d\n",
            r
        );

        return 1;
    }

    t1 = clock();

    double gpu_time =
        (double)(t1 - t0)
        / CLOCKS_PER_SEC;

    /* ========================================================
       GPU -> HOST
       ======================================================== */

    t0 = clock();

    r = cuMemcpyDtoH(
        C,
        dC,
        bytes
    );

    if (r != CUDA_SUCCESS) {

        printf(
            "cuMemcpyDtoH FAILED: %d\n",
            r
        );

        return 1;
    }

    t1 = clock();

    double d2h_time =
        (double)(t1 - t0)
        / CLOCKS_PER_SEC;

    /* ========================================================
       VERIFICATION
       
       Expected:
       
           2 * 3 + 2 = 8
       ======================================================== */

    long errors = 0;

    for (long i = 0; i < N; i++) {

        if (fabs(C[i] - 8.0) > 1e-12) {

            errors++;

            if (errors < 5) {

                printf(
                    "ERROR at %ld: "
                    "got %.15e expected 8\n",
                    i,
                    C[i]
                );
            }
        }
    }

    /* ========================================================
       RESULTS
       ======================================================== */

    printf("\n");
    printf("========================================\n");
    printf("GPU COMPUTATION TEST\n");
    printf("========================================\n");

    printf("N              : %ld\n", N);
    printf("GPU            : %s\n", name);
    printf("Block size     : %u\n", block_size);
    printf("Grid size      : %u\n", grid_size);

    printf(
        "H -> GPU       : %.6f s\n",
        h2d_time
    );

    printf(
        "GPU kernel     : %.6f s\n",
        gpu_time
    );

    printf(
        "GPU -> H       : %.6f s\n",
        d2h_time
    );

    printf(
        "Verification   : %s\n",
        errors == 0 ? "PASS" : "FAIL"
    );

    printf("========================================\n");

    /* ========================================================
       CLEANUP
       ======================================================== */

    cuMemFree(dA);
    cuMemFree(dB);
    cuMemFree(dC);

    cuCtxDestroy(ctx);

    dlclose(nvrtc_lib);
    dlclose(cuda_lib);

    free(A);
    free(B);
    free(C);

    return errors ? 1 : 0;
}


/* ============================================================
   RUN
   ============================================================ */

gpu_compute_test();

In [ ]:
#include <stdio.h>
#include <dlfcn.h>

int test_cuda_driver()
{
    void *cuda = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda) {
        printf("CUDA DRIVER: FAIL\n");
        printf("%s\n", dlerror());
        return 1;
    }

    printf("CUDA DRIVER: PASS\n");
    printf("libcuda.so.1 loaded successfully.\n");

    dlclose(cuda);

    return 0;
}

test_cuda_driver();

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <dlfcn.h>

/* CUDA Driver API types */
typedef int CUresult;
typedef int CUdevice;
typedef void *CUcontext;
typedef void *CUmodule;
typedef void *CUfunction;
typedef unsigned long long CUdeviceptr;

#define CUDA_SUCCESS 0

/* CUDA Driver functions */
typedef CUresult (*cuInit_t)(unsigned int);
typedef CUresult (*cuDeviceGetCount_t)(int *);
typedef CUresult (*cuDeviceGet_t)(CUdevice *, int);
typedef CUresult (*cuDeviceGetName_t)(char *, int, CUdevice);
typedef CUresult (*cuCtxCreate_t)(CUcontext *, unsigned int, CUdevice);
typedef CUresult (*cuCtxDestroy_t)(CUcontext);
typedef CUresult (*cuModuleLoadData_t)(CUmodule *, const void *);
typedef CUresult (*cuModuleGetFunction_t)(CUfunction *, CUmodule, const char *);
typedef CUresult (*cuLaunchKernel_t)(
    CUfunction,
    unsigned int, unsigned int, unsigned int,
    unsigned int, unsigned int, unsigned int,
    unsigned int,
    void *,
    void **,
    void *
);
typedef CUresult (*cuCtxSynchronize_t)(void);
typedef CUresult (*cuMemAlloc_t)(CUdeviceptr *, size_t);
typedef CUresult (*cuMemFree_t)(CUdeviceptr);
typedef CUresult (*cuMemcpyHtoD_t)(CUdeviceptr, const void *, size_t);
typedef CUresult (*cuMemcpyDtoH_t)(void *, CUdeviceptr, size_t);

/* NVRTC */
typedef int nvrtcResult;
typedef void *nvrtcProgram;

#define NVRTC_SUCCESS 0

typedef nvrtcResult (*nvrtcCreateProgram_t)(
    nvrtcProgram *,
    const char *,
    const char *,
    int,
    const char *const *,
    const char *const *
);

typedef nvrtcResult (*nvrtcCompileProgram_t)(
    nvrtcProgram,
    int,
    const char *const *
);

typedef nvrtcResult (*nvrtcGetPTXSize_t)(
    nvrtcProgram,
    size_t *
);

typedef nvrtcResult (*nvrtcGetPTX_t)(
    nvrtcProgram,
    char *
);

typedef nvrtcResult (*nvrtcGetProgramLogSize_t)(
    nvrtcProgram,
    size_t *
);

typedef nvrtcResult (*nvrtcGetProgramLog_t)(
    nvrtcProgram,
    char *
);

typedef nvrtcResult (*nvrtcDestroyProgram_t)(
    nvrtcProgram *
);


int gpu_nvrtc_test()
{
    /* =========================================================
       1. LOAD LIBRARIES
       ========================================================= */

    void *cuda = dlopen("libcuda.so.1", RTLD_NOW);

    if (!cuda) {
        printf("CUDA DRIVER: FAIL\n");
        printf("%s\n", dlerror());
        return 1;
    }

    void *nvrtc = dlopen(
        "/srv/conda/envs/notebook/lib/python3.11/"
        "site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12",
        RTLD_NOW
    );

    if (!nvrtc) {
        printf("NVRTC: FAIL\n");
        printf("%s\n", dlerror());
        dlclose(cuda);
        return 1;
    }

    /* =========================================================
       2. LOAD CUDA FUNCTIONS
       ========================================================= */

    cuInit_t cuInit =
        (cuInit_t)dlsym(cuda, "cuInit");

    cuDeviceGetCount_t cuDeviceGetCount =
        (cuDeviceGetCount_t)dlsym(cuda, "cuDeviceGetCount");

    cuDeviceGet_t cuDeviceGet =
        (cuDeviceGet_t)dlsym(cuda, "cuDeviceGet");

    cuDeviceGetName_t cuDeviceGetName =
        (cuDeviceGetName_t)dlsym(cuda, "cuDeviceGetName");

    cuCtxCreate_t cuCtxCreate =
        (cuCtxCreate_t)dlsym(cuda, "cuCtxCreate_v2");

    cuCtxDestroy_t cuCtxDestroy =
        (cuCtxDestroy_t)dlsym(cuda, "cuCtxDestroy_v2");

    cuModuleLoadData_t cuModuleLoadData =
        (cuModuleLoadData_t)dlsym(cuda, "cuModuleLoadData");

    cuModuleGetFunction_t cuModuleGetFunction =
        (cuModuleGetFunction_t)dlsym(cuda, "cuModuleGetFunction");

    cuLaunchKernel_t cuLaunchKernel =
        (cuLaunchKernel_t)dlsym(cuda, "cuLaunchKernel");

    cuCtxSynchronize_t cuCtxSynchronize =
        (cuCtxSynchronize_t)dlsym(cuda, "cuCtxSynchronize");

    cuMemAlloc_t cuMemAlloc =
        (cuMemAlloc_t)dlsym(cuda, "cuMemAlloc_v2");

    cuMemFree_t cuMemFree =
        (cuMemFree_t)dlsym(cuda, "cuMemFree_v2");

    cuMemcpyHtoD_t cuMemcpyHtoD =
        (cuMemcpyHtoD_t)dlsym(cuda, "cuMemcpyHtoD_v2");

    cuMemcpyDtoH_t cuMemcpyDtoH =
        (cuMemcpyDtoH_t)dlsym(cuda, "cuMemcpyDtoH_v2");

    if (!cuInit ||
        !cuDeviceGetCount ||
        !cuDeviceGet ||
        !cuDeviceGetName ||
        !cuCtxCreate ||
        !cuCtxDestroy ||
        !cuModuleLoadData ||
        !cuModuleGetFunction ||
        !cuLaunchKernel ||
        !cuCtxSynchronize ||
        !cuMemAlloc ||
        !cuMemFree ||
        !cuMemcpyHtoD ||
        !cuMemcpyDtoH) {

        printf("CUDA function loading: FAIL\n");

        dlclose(nvrtc);
        dlclose(cuda);

        return 1;
    }

    /* =========================================================
       3. LOAD NVRTC FUNCTIONS
       ========================================================= */

    nvrtcCreateProgram_t nvrtcCreateProgram =
        (nvrtcCreateProgram_t)
        dlsym(nvrtc, "nvrtcCreateProgram");

    nvrtcCompileProgram_t nvrtcCompileProgram =
        (nvrtcCompileProgram_t)
        dlsym(nvrtc, "nvrtcCompileProgram");

    nvrtcGetPTXSize_t nvrtcGetPTXSize =
        (nvrtcGetPTXSize_t)
        dlsym(nvrtc, "nvrtcGetPTXSize");

    nvrtcGetPTX_t nvrtcGetPTX =
        (nvrtcGetPTX_t)
        dlsym(nvrtc, "nvrtcGetPTX");

    nvrtcGetProgramLogSize_t nvrtcGetProgramLogSize =
        (nvrtcGetProgramLogSize_t)
        dlsym(nvrtc, "nvrtcGetProgramLogSize");

    nvrtcGetProgramLog_t nvrtcGetProgramLog =
        (nvrtcGetProgramLog_t)
        dlsym(nvrtc, "nvrtcGetProgramLog");

    nvrtcDestroyProgram_t nvrtcDestroyProgram =
        (nvrtcDestroyProgram_t)
        dlsym(nvrtc, "nvrtcDestroyProgram");

    if (!nvrtcCreateProgram ||
        !nvrtcCompileProgram ||
        !nvrtcGetPTXSize ||
        !nvrtcGetPTX ||
        !nvrtcGetProgramLogSize ||
        !nvrtcGetProgramLog ||
        !nvrtcDestroyProgram) {

        printf("NVRTC function loading: FAIL\n");

        dlclose(nvrtc);
        dlclose(cuda);

        return 1;
    }

    printf("Libraries: PASS\n");

    /* =========================================================
       4. INITIALIZE CUDA
       ========================================================= */

    if (cuInit(0) != CUDA_SUCCESS) {
        printf("cuInit: FAIL\n");
        return 1;
    }

    int count = 0;

    if (cuDeviceGetCount(&count) != CUDA_SUCCESS) {
        printf("Device count: FAIL\n");
        return 1;
    }

    printf("CUDA devices: %d\n", count);

    CUdevice device;

    if (cuDeviceGet(&device, 0) != CUDA_SUCCESS) {
        printf("Device selection: FAIL\n");
        return 1;
    }

    char name[256];

    cuDeviceGetName(
        name,
        sizeof(name),
        device
    );

    printf("GPU: %s\n", name);

    /* =========================================================
       5. CREATE CUDA CONTEXT
       ========================================================= */

    CUcontext context;

    if (cuCtxCreate(
            &context,
            0,
            device
        ) != CUDA_SUCCESS) {

        printf("CUDA context: FAIL\n");
        return 1;
    }

    printf("CUDA context: PASS\n");

    /* =========================================================
       6. CUDA KERNEL
       
       Every GPU thread calculates:

           C[i] = A[i] + B[i]

       ========================================================= */

    const char *source =
        "extern \"C\" __global__ "
        "void add_kernel("
        "const double *A, "
        "const double *B, "
        "double *C)"
        "{"
        "    int i = blockIdx.x * blockDim.x + threadIdx.x;"
        "    C[i] = A[i] + B[i];"
        "}";

    /* =========================================================
       7. COMPILE WITH NVRTC
       ========================================================= */

    nvrtcProgram program;

    if (nvrtcCreateProgram(
            &program,
            source,
            "add.cu",
            0,
            NULL,
            NULL
        ) != NVRTC_SUCCESS) {

        printf("NVRTC create: FAIL\n");
        return 1;
    }

    const char *options[] = {
        "--gpu-architecture=compute_80"
    };

    nvrtcResult result =
        nvrtcCompileProgram(
            program,
            1,
            options
        );

    if (result != NVRTC_SUCCESS) {

        size_t log_size = 0;

        nvrtcGetProgramLogSize(
            program,
            &log_size
        );

        char *log =
            malloc(log_size + 1);

        nvrtcGetProgramLog(
            program,
            log
        );

        log[log_size] = '\0';

        printf("NVRTC COMPILE: FAIL\n");
        printf("%s\n", log);

        free(log);

        nvrtcDestroyProgram(&program);
        cuCtxDestroy(context);

        dlclose(nvrtc);
        dlclose(cuda);

        return 1;
    }

    printf("NVRTC compilation: PASS\n");

    /* =========================================================
       8. GET PTX
       ========================================================= */

    size_t ptx_size = 0;

    nvrtcGetPTXSize(
        program,
        &ptx_size
    );

    char *ptx =
        malloc(ptx_size);

    nvrtcGetPTX(
        program,
        ptx
    );

    nvrtcDestroyProgram(&program);

    /* =========================================================
       9. LOAD PTX
       ========================================================= */

    CUmodule module;

    if (cuModuleLoadData(
            &module,
            ptx
        ) != CUDA_SUCCESS) {

        printf("PTX loading: FAIL\n");

        free(ptx);
        cuCtxDestroy(context);

        dlclose(nvrtc);
        dlclose(cuda);

        return 1;
    }

    free(ptx);

    printf("PTX loading: PASS\n");

    /* =========================================================
       10. GET KERNEL
       ========================================================= */

    CUfunction kernel;

    if (cuModuleGetFunction(
            &kernel,
            module,
            "add_kernel"
        ) != CUDA_SUCCESS) {

        printf("Kernel lookup: FAIL\n");
        return 1;
    }

    printf("Kernel lookup: PASS\n");

    /* =========================================================
       11. SMALL TEST ARRAY
       ========================================================= */

    const int N = 1024;

    double *A = malloc(N * sizeof(double));
    double *B = malloc(N * sizeof(double));
    double *C = malloc(N * sizeof(double));

    for (int i = 0; i < N; i++) {
        A[i] = 2.0;
        B[i] = 3.0;
        C[i] = 0.0;
    }

    CUdeviceptr dA;
    CUdeviceptr dB;
    CUdeviceptr dC;

    size_t bytes =
        N * sizeof(double);

    cuMemAlloc(&dA, bytes);
    cuMemAlloc(&dB, bytes);
    cuMemAlloc(&dC, bytes);

    cuMemcpyHtoD(
        dA,
        A,
        bytes
    );

    cuMemcpyHtoD(
        dB,
        B,
        bytes
    );

    /* =========================================================
       12. LAUNCH GPU KERNEL
       ========================================================= */

    unsigned int block = 256;
    unsigned int grid = 4;

    void *args[] = {
        &dA,
        &dB,
        &dC
    };

    if (cuLaunchKernel(
            kernel,
            grid, 1, 1,
            block, 1, 1,
            0,
            NULL,
            args,
            NULL
        ) != CUDA_SUCCESS) {

        printf("GPU kernel launch: FAIL\n");
        return 1;
    }

    cuCtxSynchronize();

    printf("GPU kernel execution: PASS\n");

    /* =========================================================
       13. COPY RESULT BACK
       ========================================================= */

    cuMemcpyDtoH(
        C,
        dC,
        bytes
    );

    /* =========================================================
       14. VERIFY
       
       2 + 3 = 5
       ========================================================= */

    int errors = 0;

    for (int i = 0; i < N; i++) {

        if (fabs(C[i] - 5.0) > 1e-12) {
            errors++;
        }
    }

    printf("\n");
    printf("====================================\n");
    printf("CUDA + NVRTC TEST\n");
    printf("====================================\n");
    printf("GPU              : %s\n", name);
    printf("N                : %d\n", N);
    printf("Expected C[i]    : 5.0\n");
    printf("Verification      : %s\n",
           errors == 0 ? "PASS" : "FAIL");
    printf("====================================\n");

    /* =========================================================
       15. CLEANUP
       ========================================================= */

    cuMemFree(dA);
    cuMemFree(dB);
    cuMemFree(dC);

    cuCtxDestroy(context);

    dlclose(nvrtc);
    dlclose(cuda);

    free(A);
    free(B);
    free(C);

    return errors ? 1 : 0;
}

gpu_nvrtc_test();

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main5(void) {
    system("nvidia-smi");
    system("ldconfig -p | grep -E 'cusolver|cublas|nvrtc' | head -30");
    return 0;
}
main5()